# Dataset Diagnostics and Exploration
This notebook contains the code to print dataset shapes, missing values, label encoder mappings, and to plot XGBoost feature importances.

In [ ]:
import os
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Raw Dataset Info

In [ ]:
train_path = r"d:\Final\Festivals\DataSet\train.csv"
festivals_path = r"d:\Final\Festivals\DataSet\festivals_base.xlsx"

if os.path.exists(train_path):
    train_df = pd.read_csv(train_path)
    print(f"Raw Sales Data Shape: {train_df.shape}")
    print(f"Raw Sales Columns: {list(train_df.columns)}")
    display(train_df.head())
else:
    print(f"Raw Sales Data not found at {train_path}")

if os.path.exists(festivals_path):
    festivals_df = pd.read_excel(festivals_path)
    print(f"Raw Festivals Data Shape: {festivals_df.shape}")
    print(f"Raw Festivals Columns: {list(festivals_df.columns)}")
    display(festivals_df.head())
else:
    print(f"Raw Festivals Data not found at {festivals_path}")

## 2. Cleaned Dataset Info

In [ ]:
cleaned_path = r"d:\Final\Festivals\DataSet\cleaned_data.csv"

if os.path.exists(cleaned_path):
    cleaned_df = pd.read_csv(cleaned_path)
    print(f"Cleaned Dataset Shape: {cleaned_df.shape}")
    print("\nCleaned Dataset Missing Values Count:")
    print(cleaned_df.isnull().sum())
    print("\nCleaned Dataset Preview (First 5 rows):")
    display(cleaned_df.head())
else:
    print(f"Cleaned dataset not found at {cleaned_path}")

## 3. Label Encoder Mappings

In [ ]:
encoders_pkl_path = r"d:\Final\Festivals\Model\label_encoders.pkl"

if os.path.exists(encoders_pkl_path):
    print("Label Encoder Mappings:")
    with open(encoders_pkl_path, 'rb') as f:
        encoders = pickle.load(f)
    for col, le in encoders.items():
        print(f"\nMappings for column '{col}':")
        for idx, val in enumerate(le.classes_):
            print(f"  {val} -> {idx}")
else:
    print(f"Label encoders not found at {encoders_pkl_path}")

## 4. XGBoost Feature Importance Plot

In [ ]:
model_pkl_path = r"d:\Final\Festivals\backend\XGmodel.pkl"
output_plot_path = r"d:\Final\Festivals\Model\feature_importance.png"

if os.path.exists(model_pkl_path) and os.path.exists(cleaned_path):
    print("Generating XGBoost Feature Importances Plot...")
    
    with open(model_pkl_path, 'rb') as f:
        model = pickle.load(f)
        
    cleaned_df = pd.read_csv(cleaned_path)
    X = cleaned_df.drop(["sales"], axis=1)
    
    importances = model.feature_importances_
    feature_imp_df = pd.DataFrame({
        'Feature': X.columns,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)
    
    # Plotting
    plt.figure(figsize=(12, 6))
    sns.barplot(x='Importance', y='Feature', data=feature_imp_df, palette='viridis')
    plt.title('XGBoost Feature Importance')
    plt.xlabel('Importance Score')
    plt.ylabel('Features')
    plt.tight_layout()
    
    # Show the plot inline in the notebook
    plt.show()
    
    # Save the plot as well
    os.makedirs(os.path.dirname(output_plot_path), exist_ok=True)
    plt.savefig(output_plot_path)
    print(f"Plot saved successfully to: {output_plot_path}")
else:
    print("Trained model or cleaned dataset not found. Skipping feature importance plot.")